In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import lightgbm as gbm
import joblib

In [13]:
df = pd.read_csv(r'C:\Users\Нуридин\Desktop\churn-project\Data\WA_Fn-UseC_-Telco-Customer-Churn.csv')


In [14]:
df.head(15)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [15]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [16]:
df_cup = df.copy()
numeric_cols = df_cup.select_dtypes(include=[np.number]).columns
def cap_outliers_iqr(df: pd.DataFrame, k: float = 1.5) -> pd.DataFrame:
    """
    Возвращает копию df, где все числовые колонки ограничены по IQR-методу.
    Значения за пределами [Q1 - k*IQR, Q3 + k*IQR] обрезаются до границ.
    """
    
    for col in numeric_cols:
        Q1 = df_cup[col].quantile(0.1)
        Q3 = df_cup[col].quantile(0.9)
        IQR = Q3 - Q1
        lower = Q1 - k * IQR
        upper = Q3 + k * IQR
        df_cup[col] = df_cup[col].clip(lower, upper)
    return df_cup

In [17]:
df_clean = cap_outliers_iqr(df, k=2)
df_clean.head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [18]:
X = df_clean.drop(columns=["Churn","customerID"])
y = df_clean["Churn"].map({"No": 0, "Yes": 1})

In [19]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size = 0.2, 
    random_state=42,
    stratify = y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

In [20]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns

In [21]:
print("Числовые:", numeric_cols.tolist())
print("Категориальные:", categorical_cols.tolist())

Числовые: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Категориальные: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [22]:
ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

ohe.fit(X_train[categorical_cols])

X_train_cat = ohe.transform(X_train[categorical_cols])
X_val_cat = ohe.transform(X_val[categorical_cols])
X_test_cat = ohe.transform(X_test[categorical_cols])

X_train_num = X_train[numeric_cols].to_numpy()
X_val_num = X_val[numeric_cols].to_numpy()
X_test_num = X_test[numeric_cols].to_numpy()


In [23]:
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = list(numeric_cols) + list(cat_feature_names)

In [24]:
X_train_final = pd.DataFrame(
    np.hstack([X_train_num, X_train_cat]),
    columns=feature_names,
    index=X_train.index
)

X_val_final = pd.DataFrame(
    np.hstack([X_val_num, X_val_cat]),
    columns=feature_names,
    index=X_val.index
)

X_test_final = pd.DataFrame(
    np.hstack([X_test_num, X_test_cat]),
    columns=feature_names,
    index=X_test.index
)

X_train_final = X_train_final.fillna(X_train_final.median(numeric_only=True))
X_val_final   = X_val_final.fillna(X_train_final.median(numeric_only=True))
X_test_final  = X_test_final.fillna(X_train_final.median(numeric_only=True))

In [25]:
print("Размеры:")
print("train:", X_train_final.shape)
print("val:  ", X_val_final.shape)
print("test: ", X_test_final.shape)

print("\nПервые имена признаков:")
print(X_train_final.columns[:].tolist())
print(len(X_train_final.columns))

total_nan = X_test_final.isna().sum().sum()
print("Всего NaN:", total_nan)

Размеры:
train: (4225, 45)
val:   (1409, 45)
test:  (1409, 45)

Первые имена признаков:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_No', '

In [26]:
def calc_metrics(y_true, y_pred, y_pred_proba=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }
    if y_pred_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
    return metrics

In [27]:
models = {
    'LogReg': LogisticRegression(
        max_iter=1000,
        C=1.0,
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
    'LightGBM': gbm.LGBMClassifier(
        n_estimators = 300,
        learning_rate=0.1,
        max_depth=-1,
        random_state=42,
        n_jobs=-1       
    )
}

In [28]:
results = {}

for name, model in models.items():
    print(f"=== {name} ===")
    
    model.fit(X_train_final, y_train)
    
    y_pred_val = model.predict(X_val_final)
    y_pred_proba_val = model.predict_proba(X_val_final)[:, 1]
    
    y_pred_test = model.predict(X_test_final)
    y_pred_proba_test = model.predict_proba(X_test_final)[:, 1]
                                                          
    metrics_val = calc_metrics(y_val, y_pred_val, y_pred_proba_val)
    metrics_test = calc_metrics(y_test, y_pred_test, y_pred_proba_test)
    
    results[name] = {
        'model': model,
        'val': metrics_val,
        'test': metrics_test     
    }   
print("val_metrics", metrics_val)    
print("test_metrics", metrics_test)

=== LogReg ===


c:\Users\Нуридин\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== RandomForest ===
=== LightGBM ===
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1121, number of negative: 3104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000903 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 667
[LightGBM] [Info] Number of data points in the train set: 4225, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265325 -> initscore=-1.018470
[LightGBM] [Info] Start training from score -1.018470
val_metrics {'accuracy': 0.7764371894960965, 'f1': 0.5387994143484627, 'roc_auc': 0.814800175669741}
test_metrics {'accuracy': 0.7920511000709723, 'f1': 0.5672082717872969, 'roc_auc': 0.821115244516779}


In [29]:
rows = []
for name, res in results.items():
    row = {"model": name}
    row.update({f"val_{k}": v for k, v in res["val"].items()})
    row.update({f"test_{k}": v for k, v in res["test"].items()})
    rows.append(row)

metrics_df = pd.DataFrame(rows)
print(metrics_df)

          model  val_accuracy    val_f1  val_roc_auc  test_accuracy   test_f1  \
0        LogReg      0.802697  0.589971     0.835897       0.803407  0.599132   
1  RandomForest      0.780696  0.523883     0.813595       0.781405  0.540299   
2      LightGBM      0.776437  0.538799     0.814800       0.792051  0.567208   

   test_roc_auc  
0      0.843445  
1      0.820649  
2      0.821115  


In [30]:
from sklearn.metrics import f1_score
import numpy as np

proba_val = model.predict_proba(X_val_final)[:, 1]

thresholds = np.linspace(0.35, 0.65, 31)
best_thr, best_f1 = 0.5, 0

for thr in thresholds:
    y_pred_thr = (proba_val >= thr).astype(int)
    f1 = f1_score(y_val, y_pred_thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print("Best threshold:", best_thr, "F1 val:", best_f1)

y_pred_test = (model.predict_proba(X_test_final)[:, 1] >= best_thr).astype(int)
f1_test = f1_score(y_test, y_pred_test)
print("F1 test с новым порогом:", f1_test)

Best threshold: 0.35 F1 val: 0.5914110429447853
F1 test с новым порогом: 0.6115288220551378


In [31]:
model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_final, y_train)
proba = model.predict_proba(X_train_final)[:5, 1]
print("Пример вероятностей:", proba)

Пример вероятностей: [0.12243917 0.88408629 0.2629963  0.01808879 0.72637503]


In [32]:

THRESHOLD = 0.35

joblib.dump(
    {"model": model, "threshold": THRESHOLD},
    "churn_model_v1.joblib"
)

print("Модель сохранена в churn_model_v1.joblib")

Модель сохранена в churn_model_v1.joblib


In [33]:
import joblib
import pandas as pd

_bundle = None

def _get_model():
    global _bundle
    if _bundle is None:
        _bundle = joblib.load("churn_model_v1.joblib")
    return _bundle["model"], _bundle["threshold"]

def predict_churn(df: pd.DataFrame):
    model, THRESHOLD = _get_model()
    
    proba = model.predict_proba(df)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    
    return proba, pred

In [34]:
'''import joblib
import pandas as pd

_bundle = None

def _get_model():
    global _bundle
    if _bundle is None:
        _bundle = joblib.load("churn_model_v1.joblib")
    return _bundle["model"], _bundle["threshold"]

def predict_churn(df: pd.DataFrame):
    model, THRESHOLD = _get_model()
    
    proba = model.predict_proba(df)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    
    return proba, pred'''

'import joblib\nimport pandas as pd\n\n_bundle = None\n\ndef _get_model():\n    global _bundle\n    if _bundle is None:\n        _bundle = joblib.load("churn_model_v1.joblib")\n    return _bundle["model"], _bundle["threshold"]\n\ndef predict_churn(df: pd.DataFrame):\n    model, THRESHOLD = _get_model()\n\n    proba = model.predict_proba(df)[:, 1]\n    pred = (proba >= THRESHOLD).astype(int)\n\n    return proba, pred'

In [35]:
bundle = joblib.load("churn_model_v1.joblib")

print("Ключи в файле:", bundle.keys())

# Если там модель и порог:
model = bundle["model"]
threshold = bundle["threshold"]

print("Порог:", threshold)
print("Тип модели:", type(model))
print("Модель:\n", model)

Ключи в файле: dict_keys(['model', 'threshold'])
Порог: 0.35
Тип модели: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Модель:
 LogisticRegression(class_weight='balanced', max_iter=3000, random_state=42)


In [36]:
import joblib

joblib.dump(ohe, "models/ohe.joblib")
joblib.dump({
    "numeric_cols": list(numeric_cols),
    "categorical_cols": list(categorical_cols),
    "feature_names": list(feature_names),
}, "models/feature_info.joblib")

['models/feature_info.joblib']